In [23]:
import pandas as pd
import numpy as np
import os

# Load the datasets
df_players = pd.read_csv("to_merge_data/j2_players.csv")
df_appearances = pd.read_csv("to_merge_data/j2_appearances.csv")
df_clubs = pd.read_csv("to_merge_data/j2_clubs.csv")
df_val = pd.read_csv("to_merge_data/j2_val.csv")

In [24]:
#merge appearances with players and clubs
merge_df = pd.merge(df_appearances, df_players, on='player_id', how='left')
final_merge_df = pd.merge(merge_df, df_clubs, on='club_id', how='left')
final_merge_df.head()

,player_id,player_name,season,first_game_date,club_id,minutes_played,goals,assists,yellow_cards,red_cards,last_game_date,name_x,date_of_birth,position,market_value_in_eur,country_of_citizenship,foot,height_in_cm,name_y,domestic_competition_id
0,10,Miroslav Klose,12/13,2012-08-23,398,2585,16,3,8,0,2013-06-30,Miroslav Klose,1978-06-09,Attack,1000000.0,Germany,Right,184.0,Società Sportiva Lazio S.P.A.,IT1
1,10,Miroslav Klose,13/14,2013-08-18,398,2220,8,5,2,0,2014-06-30,Miroslav Klose,1978-06-09,Attack,1000000.0,Germany,Right,184.0,Società Sportiva Lazio S.P.A.,IT1
2,10,Miroslav Klose,14/15,2014-08-24,398,2289,16,9,6,0,2015-06-30,Miroslav Klose,1978-06-09,Attack,1000000.0,Germany,Right,184.0,Società Sportiva Lazio S.P.A.,IT1
3,10,Miroslav Klose,15/16,2015-08-08,398,1714,8,8,3,0,2016-06-30,Miroslav Klose,1978-06-09,Attack,1000000.0,Germany,Right,184.0,Società Sportiva Lazio S.P.A.,IT1
4,26,Roman Weidenfeller,12/13,2012-08-12,16,4401,0,0,2,1,2013-06-30,Roman Weidenfeller,1980-08-06,Goalkeeper,750000.0,Germany,Left,190.0,Borussia Dortmund,L1


In [ ]:
# drop unnecessary columns
final_merge_df = final_merge_df.drop(['name_x'], axis=1)
#rename columns for clarity
final_merge_df =final_merge_df.rename(columns={'name_y': 'club_name'})

final_merge_df['last_game_date'] = pd.to_datetime(final_merge_df['last_game_date'], errors='coerce')
final_merge_df['date_of_birth'] = pd.to_datetime(final_merge_df['date_of_birth'], errors='coerce')

final_merge_df['age'] = final_merge_df['last_game_date'].dt.year - final_merge_df['date_of_birth'].dt.year - (
    (final_merge_df['last_game_date'].dt.month < final_merge_df['date_of_birth'].dt.month) |
    ((final_merge_df['last_game_date'].dt.month == final_merge_df['date_of_birth'].dt.month) &
     (final_merge_df['last_game_date'].dt.day < final_merge_df['date_of_birth'].dt.day))
)
#reorder columns for better readability
final_merge_df = final_merge_df[['player_id', 'player_name', 'country_of_citizenship',
                                 'foot', 'height_in_cm', 'season',
                                 'first_game_date','last_game_date', 'club_id',
                                 'club_name', 'domestic_competition_id', 'age',
                                 'position', 'minutes_played', 'goals',
                                 'assists', 'yellow_cards', 'red_cards']]

In [26]:
value_season_df = pd.merge(final_merge_df, df_val, on=['player_id', 'season'], how='left')

In [27]:
# Sort by player_id and season to ensure proper order
value_season_df = value_season_df.sort_values(['player_id', 'season'])

# Convert season string "12/13" into a year
def season_to_year(s):
    start_year = int(s.split('/')[0])
    return 2000 + start_year

value_season_df['season_year'] = value_season_df['season'].apply(season_to_year)

# Sort again
value_season_df = value_season_df.sort_values(['player_id', 'season_year'])

# Shift market_value_to the previous known value for each player
value_season_df['prev_season_market_value'] = value_season_df.groupby('player_id')['market_value_in_eur'].shift(1)

# Fill NaNs with previous season market value
value_season_df['market_value_in_eur'] = value_season_df['market_value_in_eur'].fillna(value_season_df['prev_season_market_value'])

# Drop helper colummns
value_season_df.drop(columns=['season_year', 'prev_season_market_value'], inplace=True)

In [ ]:
#sorting the dataframe for better comprehension
final_version_df = value_season_df.groupby(['player_id', 'player_name', 'country_of_citizenship',
                                 'foot', 'height_in_cm', 'season',
                                 'first_game_date','last_game_date', 'club_id',
                                 'club_name', 'domestic_competition_id', 'age',
                                 'position', 'minutes_played', 'goals',
                                 'assists', 'yellow_cards', 'red_cards']).mean('market_value_in_eur').reset_index()

final_version_df.head(20)

,player_id,player_name,country_of_citizenship,foot,height_in_cm,season,first_game_date,last_game_date,club_id,club_name,domestic_competition_id,age,position,minutes_played,goals,assists,yellow_cards,red_cards,market_value_in_eur
0,10,Miroslav Klose,Germany,Right,184.0,12/13,2012-08-23,2013-06-30,398,Società Sportiva Lazio S.P.A.,IT1,35.0,Attack,2585,16,3,8,0,4.000000e+06
1,10,Miroslav Klose,Germany,Right,184.0,13/14,2013-08-18,2014-06-30,398,Società Sportiva Lazio S.P.A.,IT1,36.0,Attack,2220,8,5,2,0,1.000000e+06
2,10,Miroslav Klose,Germany,Right,184.0,14/15,2014-08-24,2015-06-30,398,Società Sportiva Lazio S.P.A.,IT1,37.0,Attack,2289,16,9,6,0,1.000000e+06
3,10,Miroslav Klose,Germany,Right,184.0,15/16,2015-08-08,2016-06-30,398,Società Sportiva Lazio S.P.A.,IT1,38.0,Attack,1714,8,8,3,0,1.000000e+06
4,26,Roman Weidenfeller,Germany,Left,190.0,12/13,2012-08-12,2013-06-30,16,Borussia Dortmund,L1,32.0,Goalkeeper,4401,0,0,2,1,4.500000e+06
5,26,Roman Weidenfeller,Germany,Left,190.0,13/14,2013-07-27,2014-06-30,16,Borussia Dortmund,L1,33.0,Goalkeeper,3855,0,0,1,1,5.000000e+06
6,26,Roman Weidenfeller,Germany,Left,190.0,14/15,2014-08-29,2015-06-30,16,Borussia Dortmund,L1,34.0,Goalkeeper,2880,0,0,0,0,4.000000e+06
7,26,Roman Weidenfeller,Germany,Left,190.0,15/16,2015-08-06,2016-06-30,16,Borussia Dortmund,L1,35.0,Goalkeeper,1260,0,0,1,0,1.333333e+06
8,26,Roman Weidenfeller,Germany,Left,190.0,16/17,2016-08-22,2017-06-30,16,Borussia Dortmund,L1,36.0,Goalkeeper,1020,0,0,0,0,1.000000e+06
9,26,Roman Weidenfeller,Germany,Left,190.0,17/18,2017-11-21,2018-06-30,16,Borussia Dortmund,L1,37.0,Goalkeeper,92,0,0,0,0,7.500000e+05


In [29]:
import os

# create the directory
os.makedirs("complete_dataset", exist_ok=True)

In [30]:
final_merge_df.to_csv("complete_dataset/j_complete_dataset.csv", index=False)